# 文本分类模型训练工具集 - Google Colab GPU版本

## 📋 目录
本Notebook包含多个文本分类模型训练脚本，整合为单一文件便于在Google Colab中使用GPU运行。

### 模型列表
1. **基础LSTM分类器** (`train_scratch_lstm.py`) - 基础双向LSTM文本分类
2. **ULMFiT模型** (`train_ulmfit_scratch.py`) - ULMFiT预训练+微调
3. **强LSTM模型** (`train_strong_lstm.py`) - 多层BiLSTM+CNN特征
4. **ELMo风格模型** (`train_elmo_scratch.py`) - 双向语言模型融合
5. **n-gram混合模型** (`train_ulmfit_ngram.py`) - ULMFiT+n-gram特征
6. **VAT半监督模型** (`train_ulmfit_vat.py`) - 虚拟对抗训练
7. **自训练模型** (`selftrain_from_checkpoint.py`) - 伪标签自训练
8. **继续微调** (`continue_ulmfit_finetune.py`) - 从检查点继续微调
9. **全量微调** (`finetune_all_from_checkpoint.py`) - 全参数微调
10. **生成提交** (`make_submission_from_checkpoint.py`) - 从检查点生成预测
11. **NB-LSTM混合** (`nb_lstm_hybrid.py`) - 朴素贝叶斯+LSTM集成

## ⚙️ Colab设置
1. 点击 **运行时** → **更改运行时类型** → 选择 **GPU**
2. 上传您的数据文件（train.csv, test.csv, train_unlabel.csv）
3. 根据需要修改各部分的参数并运行单元格

---


In [ ]:
# ============================================
# 第一部分：环境设置和依赖安装
# ============================================
# 功能：安装必要的依赖包，设置运行环境

# 检查GPU是否可用
import torch
print(f"PyTorch版本：{torch.__version__}")
print(f"CUDA可用：{torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备：{torch.cuda.get_device_name(0)}")
    print(f"CUDA版本：{torch.version.cuda}")

# 设置环境变量（避免某些OpenMP重复加载问题）
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

In [ ]:
# ============================================
# 第二部分：通用工具函数（来自train_scratch_lstm.py）
# ============================================
# 功能：提供文本处理、数据加载、词汇构建、LSTM模型等基础功能
# 这是所有其他模型的基础依赖模块

import argparse
import csv
import html
import math
import os
import random
import re
from collections import Counter
from dataclasses import dataclass
from typing import Iterable, List, Optional, Sequence, Tuple

# Some Windows PyTorch installs load duplicate OpenMP runtimes. This needs to be
# set before importing torch.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F


TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+|[^\w\s]", re.IGNORECASE)
PAD = "<pad>"
UNK = "<unk>"


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def normalize_text(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = text.replace("\x85", " ")
    return text.lower()


def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(normalize_text(text))


def read_train(path: str, max_rows: Optional[int] = None) -> Tuple[List[str], List[int]]:
    texts: List[str] = []
    labels: List[int] = []
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i == 0 and len(row) == 2 and row[0] == "0" and row[1] in {"0", "1"}:
                continue
            if len(row) != 2:
                continue
            texts.append(row[0])
            labels.append(int(row[1]))
            if max_rows and len(texts) >= max_rows:
                break
    return texts, labels


def read_unlabel(path: str, max_rows: Optional[int] = None) -> List[str]:
    texts: List[str] = []
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i == 0 and len(row) == 1 and row[0] == "0":
                continue
            if len(row) != 1:
                continue
            texts.append(row[0])
            if max_rows and len(texts) >= max_rows:
                break
    return texts


def read_test(path: str) -> Tuple[List[str], List[str]]:
    ids: List[str] = []
    texts: List[str] = []
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) != 2:
                continue
            ids.append(row[0])
            texts.append(row[1])
    return ids, texts


def build_vocab(
    tokenized_texts: Iterable[Sequence[str]],
    min_count: int,
    max_vocab: int,
) -> Tuple[dict, List[str]]:
    counter: Counter = Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    idx2word = [PAD, UNK]
    for word, count in counter.most_common():
        if count < min_count:
            break
        if len(idx2word) >= max_vocab:
            break
        idx2word.append(word)

    word2idx = {word: idx for idx, word in enumerate(idx2word)}
    return word2idx, idx2word


def encode(tokens: Sequence[str], word2idx: dict, seq_len: int) -> Tuple[List[int], int]:
    if len(tokens) > seq_len:
        # Keep the beginning and ending. Movie reviews often summarize sentiment at both ends.
        head = seq_len // 2
        tail = seq_len - head
        tokens = list(tokens[:head]) + list(tokens[-tail:])
    ids = [word2idx.get(token, 1) for token in tokens]
    length = len(ids)
    if length < seq_len:
        ids.extend([0] * (seq_len - length))
    return ids, max(1, length)


@dataclass
class EncodedDataset:
    x: torch.Tensor
    lengths: torch.Tensor
    y: Optional[torch.Tensor] = None
    weights: Optional[torch.Tensor] = None

    def __len__(self) -> int:
        return self.x.size(0)


class BucketBatcher:
    def __init__(
        self,
        dataset: EncodedDataset,
        batch_size: int,
        shuffle: bool,
        seed: int,
        bucket_size: int = 2048,
    ) -> None:
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.bucket_size = bucket_size
        self.epoch = 0

    def __len__(self) -> int:
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1

        if self.shuffle:
            rng.shuffle(indices)
            buckets = [
                indices[i : i + self.bucket_size]
                for i in range(0, len(indices), self.bucket_size)
            ]
            for bucket in buckets:
                bucket.sort(key=lambda idx: int(self.dataset.lengths[idx]), reverse=True)
            rng.shuffle(buckets)
            indices = [idx for bucket in buckets for idx in bucket]
        else:
            indices.sort(key=lambda idx: int(self.dataset.lengths[idx]), reverse=True)

        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start : start + self.batch_size]
            idx_tensor = torch.tensor(batch_idx, dtype=torch.long)
            fields = [
                self.dataset.x.index_select(0, idx_tensor),
                self.dataset.lengths.index_select(0, idx_tensor),
            ]
            if self.dataset.y is not None:
                fields.append(self.dataset.y.index_select(0, idx_tensor))
            if self.dataset.weights is not None:
                fields.append(self.dataset.weights.index_select(0, idx_tensor))
            yield tuple(fields)


def make_dataset(
    texts: Sequence[str],
    labels: Optional[Sequence[float]],
    word2idx: dict,
    seq_len: int,
    weights: Optional[Sequence[float]] = None,
) -> EncodedDataset:
    ids: List[List[int]] = []
    lengths: List[int] = []
    for text in texts:
        encoded, length = encode(tokenize(text), word2idx, seq_len)
        ids.append(encoded)
        lengths.append(length)

    y_tensor = None
    if labels is not None:
        y_tensor = torch.tensor(labels, dtype=torch.float32)

    w_tensor = None
    if weights is not None:
        w_tensor = torch.tensor(weights, dtype=torch.float32)

    return EncodedDataset(
        x=torch.tensor(ids, dtype=torch.long),
        lengths=torch.tensor(lengths, dtype=torch.long),
        y=y_tensor,
        weights=w_tensor,
    )


def split_indices(n: int, valid_ratio: float, seed: int) -> Tuple[List[int], List[int]]:
    indices = list(range(n))
    random.Random(seed).shuffle(indices)
    valid_n = int(n * valid_ratio)
    valid_idx = sorted(indices[:valid_n])
    train_idx = sorted(indices[valid_n:])
    return train_idx, valid_idx


def subset_dataset(dataset: EncodedDataset, indices: Sequence[int]) -> EncodedDataset:
    idx = torch.tensor(indices, dtype=torch.long)
    return EncodedDataset(
        x=dataset.x.index_select(0, idx),
        lengths=dataset.lengths.index_select(0, idx),
        y=dataset.y.index_select(0, idx) if dataset.y is not None else None,
        weights=dataset.weights.index_select(0, idx)
        if dataset.weights is not None
        else None,
    )


class ScratchLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.x2h = nn.Linear(input_dim, 4 * hidden_dim)
        self.h2h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x_t: torch.Tensor, state: Tuple[torch.Tensor, torch.Tensor]):
        return self.forward_from_input(self.x2h(x_t), state)

    def forward_from_input(
        self, x_gate_t: torch.Tensor, state: Tuple[torch.Tensor, torch.Tensor]
    ):
        h, c = state
        gates = x_gate_t + self.h2h(h)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f + 1.0)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


class ScratchBiLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.fw = ScratchLSTMCell(input_dim, hidden_dim)
        self.bw = ScratchLSTMCell(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def _run_direction(
        self,
        cell: ScratchLSTMCell,
        x: torch.Tensor,
        lengths: torch.Tensor,
        reverse: bool,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        batch, seq_len, _ = x.shape
        h = x.new_zeros(batch, self.hidden_dim)
        c = x.new_zeros(batch, self.hidden_dim)
        x_gates = cell.x2h(x)
        outputs = []
        steps = range(seq_len - 1, -1, -1) if reverse else range(seq_len)

        for t in steps:
            h_new, c_new = cell.forward_from_input(x_gates[:, t, :], (h, c))
            active = (lengths > t).to(x.dtype).unsqueeze(1)
            h = active * h_new + (1.0 - active) * h
            c = active * c_new + (1.0 - active) * c
            outputs.append(h)

        if reverse:
            outputs.reverse()
        return torch.stack(outputs, dim=1), h

    def forward(self, x: torch.Tensor, lengths: torch.Tensor):
        fw_out, fw_last = self._run_direction(self.fw, x, lengths, reverse=False)
        bw_out, bw_last = self._run_direction(self.bw, x, lengths, reverse=True)
        out = torch.cat([fw_out, bw_out], dim=2)
        out = self.dropout(out)
        last = torch.cat([fw_last, bw_last], dim=1)
        return out, last


class TextLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int,
        hidden_dim: int,
        dropout: float,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = ScratchBiLSTM(emb_dim, hidden_dim, dropout)
        out_dim = hidden_dim * 2
        self.attn = nn.Linear(out_dim, 1)
        self.fc = nn.Sequential(
            nn.LayerNorm(out_dim * 3),
            nn.Dropout(dropout),
            nn.Linear(out_dim * 3, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.05)
        with torch.no_grad():
            self.embedding.weight[pad_idx].zero_()

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.emb_dropout(self.embedding(x))
        out, last = self.lstm(emb, lengths)
        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)

        attn_score = self.attn(out).squeeze(2).masked_fill(~mask, -1e4)
        attn_weight = F.softmax(attn_score, dim=1).unsqueeze(2)
        attn_pool = torch.sum(out * attn_weight, dim=1)

        masked_out = out.masked_fill(~mask.unsqueeze(2), -1e4)
        max_pool = masked_out.max(dim=1).values

        features = torch.cat([last, attn_pool, max_pool], dim=1)
        return self.fc(features).squeeze(1)


def move_batch(batch, device):
    return tuple(item.to(device, non_blocking=True) for item in batch)


def weighted_bce(logits, labels, weights):
    loss = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def smooth_binary_labels(labels: torch.Tensor, smoothing: float) -> torch.Tensor:
    if smoothing <= 0.0:
        return labels
    return labels * (1.0 - 2.0 * smoothing) + smoothing


def evaluate(model, batcher, device) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in batcher:
            x, lengths, labels = move_batch(batch, device)
            logits = model(x, lengths)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * labels.numel()
            total_correct += (pred == labels).sum().item()
            total += labels.numel()
    return total_loss / total, total_correct / total


def train_model(
    model,
    train_data: EncodedDataset,
    valid_data: EncodedDataset,
    args,
    device,
    stage_name: str,
    lr: Optional[float] = None,
) -> float:
    train_batcher = BucketBatcher(train_data, args.batch_size, True, args.seed)
    valid_batcher = BucketBatcher(valid_data, args.batch_size, False, args.seed)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.lr if lr is None else lr,
        weight_decay=args.weight_decay,
        betas=(0.9, 0.999),
    )
    total_steps = max(1, len(train_batcher) * args.epochs)
    warmup_steps = max(1, int(total_steps * args.warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    amp_enabled = args.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_acc = -1.0
    best_path = os.path.join(args.out_dir, f"{stage_name}_best.pt")
    global_step = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for batch in train_batcher:
            if len(batch) == 4:
                x, lengths, labels, weights = move_batch(batch, device)
            else:
                x, lengths, labels = move_batch(batch, device)
                weights = torch.ones_like(labels)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                logits = model(x, lengths)
                targets = smooth_binary_labels(
                    labels, getattr(args, "label_smoothing", 0.0)
                )
                loss = weighted_bce(logits, targets, weights)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1

            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total_seen += labels.numel()
                total_loss += loss.item() * labels.numel()

        val_loss, val_acc = evaluate(model, valid_batcher, device)
        train_acc = total_correct / max(1, total_seen)
        print(
            f"{stage_name} epoch {epoch:02d} "
            f"train_loss={total_loss / max(1, total_seen):.5f} "
            f"train_acc={train_acc:.4f} val_loss={val_loss:.5f} val_acc={val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(
                {
                    "model": model.state_dict(),
                    "args": vars(args),
                    "val_acc": best_acc,
                    "stage": stage_name,
                },
                best_path,
            )
            print(f"saved {best_path} val_acc={best_acc:.4f}")

    checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    return best_acc


def predict_probs(model, dataset: EncodedDataset, batch_size: int, device) -> List[float]:
    model.eval()
    probs: List[Optional[float]] = [None] * len(dataset)
    indices = list(range(len(dataset)))
    indices.sort(key=lambda idx: int(dataset.lengths[idx]), reverse=True)
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start : start + batch_size]
            idx_tensor = torch.tensor(batch_idx, dtype=torch.long)
            x = dataset.x.index_select(0, idx_tensor).to(device, non_blocking=True)
            lengths = dataset.lengths.index_select(0, idx_tensor).to(device, non_blocking=True)
            logits = model(x, lengths)
            batch_probs = torch.sigmoid(logits).detach().cpu().tolist()
            for original_idx, prob in zip(batch_idx, batch_probs):
                probs[original_idx] = prob
    return [float(prob) for prob in probs if prob is not None]


def write_submission(ids: Sequence[str], probs: Sequence[float], path: str) -> None:
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, prob in zip(ids, probs):
            writer.writerow([sample_id, int(prob >= 0.5)])


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--unlabel", default="train_unlabel.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--out-dir", default="runs_scratch_lstm")
    parser.add_argument("--submission", default="submission.csv")
    parser.add_argument("--seq-len", type=int, default=320)
    parser.add_argument("--min-count", type=int, default=2)
    parser.add_argument("--max-vocab", type=int, default=90000)
    parser.add_argument("--emb-dim", type=int, default=256)
    parser.add_argument("--hidden-dim", type=int, default=224)
    parser.add_argument("--dropout", type=float, default=0.35)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--lr", type=float, default=2e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--warmup-ratio", type=float, default=0.08)
    parser.add_argument("--label-smoothing", type=float, default=0.0)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--self-train", action="store_true")
    parser.add_argument("--self-threshold", type=float, default=0.985)
    parser.add_argument("--pseudo-weight", type=float, default=0.35)
    parser.add_argument("--pseudo-limit", type=int, default=30000)
    parser.add_argument("--self-lr-mult", type=float, default=0.25)
    parser.add_argument("--include-test-vocab", action="store_true", default=True)
    parser.add_argument("--no-include-test-vocab", dest="include_test_vocab", action="store_false")
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-train", type=int, default=None)
    parser.add_argument("--max-unlabel", type=int, default=None)
    parser.add_argument("--skip-predict", action="store_true")
    return parser.parse_args()


def main() -> None:
    # Works around duplicate OpenMP runtimes in some Windows PyTorch installs.
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")

    train_texts, train_labels = read_train(args.train, args.max_train)
    unlabel_texts = read_unlabel(args.unlabel, args.max_unlabel)
    test_ids, test_texts = read_test(args.test)
    print(
        f"loaded train={len(train_texts)} unlabel={len(unlabel_texts)} test={len(test_texts)}"
    )

    print("tokenizing for vocabulary ...")
    vocab_texts = train_texts + unlabel_texts
    if args.include_test_vocab:
        vocab_texts = vocab_texts + test_texts
    vocab_source = [tokenize(text) for text in vocab_texts]
    word2idx, idx2word = build_vocab(vocab_source, args.min_count, args.max_vocab)
    print(f"vocab={len(idx2word)} min_count={args.min_count}")

    full_train = make_dataset(train_texts, train_labels, word2idx, args.seq_len)
    train_idx, valid_idx = split_indices(len(full_train), args.valid_ratio, args.seed)
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)

    model = TextLSTMClassifier(
        vocab_size=len(idx2word),
        emb_dim=args.emb_dim,
        hidden_dim=args.hidden_dim,
        dropout=args.dropout,
    ).to(device)

    print("training supervised model ...")
    best_acc = train_model(model, train_data, valid_data, args, device, "supervised")
    best_overall_acc = best_acc
    best_overall_path = os.path.join(args.out_dir, "supervised_best.pt")
    print(f"best supervised val_acc={best_acc:.4f}")

    if args.self_train and unlabel_texts:
        print("selecting pseudo labels ...")
        unlabel_data = make_dataset(unlabel_texts, None, word2idx, args.seq_len)
        probs = predict_probs(model, unlabel_data, args.batch_size, device)
        selected = [
            (i, prob)
            for i, prob in enumerate(probs)
            if prob >= args.self_threshold or prob <= 1.0 - args.self_threshold
        ]
        selected.sort(key=lambda item: abs(item[1] - 0.5), reverse=True)
        selected = selected[: args.pseudo_limit]
        print(f"pseudo_selected={len(selected)} threshold={args.self_threshold}")

        if selected:
            pseudo_texts = [unlabel_texts[i] for i, _ in selected]
            pseudo_labels = [1.0 if prob >= 0.5 else 0.0 for _, prob in selected]
            combined_texts = [train_texts[i] for i in train_idx] + pseudo_texts
            combined_labels = [float(train_labels[i]) for i in train_idx] + pseudo_labels
            combined_weights = [1.0] * len(train_idx) + [args.pseudo_weight] * len(
                pseudo_texts
            )
            train_plus_pseudo = make_dataset(
                combined_texts,
                combined_labels,
                word2idx,
                args.seq_len,
                combined_weights,
            )
            print("fine-tuning with pseudo labels ...")
            self_acc = train_model(
                model,
                train_plus_pseudo,
                valid_data,
                args,
                device,
                "selftrain",
                lr=args.lr * args.self_lr_mult,
            )
            print(f"best self-train val_acc={self_acc:.4f}")
            if self_acc > best_overall_acc:
                best_overall_acc = self_acc
                best_overall_path = os.path.join(args.out_dir, "selftrain_best.pt")

    best_checkpoint = torch.load(best_overall_path, map_location=device, weights_only=False)
    model.load_state_dict(best_checkpoint["model"])
    best_acc = best_overall_acc
    print(f"using checkpoint={best_overall_path} val_acc={best_acc:.4f}")

    if not args.skip_predict:
        print("predicting test ...")
        test_data = make_dataset(test_texts, None, word2idx, args.seq_len)
        test_probs = predict_probs(model, test_data, args.batch_size, device)
        write_submission(test_ids, test_probs, args.submission)
        print(f"wrote {args.submission}")

    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": vars(args),
            "val_acc": best_acc,
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第三部分：ULMFiT模型（来自train_ulmfit_scratch.py）
# ============================================
# 功能：实现ULMFiT（Universal Language Model Fine-tuning）模型
# 包括：单向LSTM编码器、语言模型预训练、分类器微调
# 使用方法：运行此单元格后，可以使用ULMFiTClassifier类进行训练

import argparse
import math
import os
import random
from typing import List, Optional, Sequence, Tuple

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    BucketBatcher,
    EncodedDataset,
    ScratchLSTMCell,
    build_vocab,
    encode,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    set_seed,
    split_indices,
    subset_dataset,
    tokenize,
    write_submission,
)


EOS = "<eos>"


class UniLSTMLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.cell = ScratchLSTMCell(input_dim, hidden_dim)
        self.hidden_dim = hidden_dim
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
        state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        batch, seq_len, _ = x.shape
        if state is None:
            h = x.new_zeros(batch, self.hidden_dim)
            c = x.new_zeros(batch, self.hidden_dim)
        else:
            h, c = state
        x_gates = self.cell.x2h(x)
        outputs = []
        for t in range(seq_len):
            h_new, c_new = self.cell.forward_from_input(x_gates[:, t, :], (h, c))
            if lengths is not None:
                active = (lengths > t).to(x.dtype).unsqueeze(1)
                h = active * h_new + (1.0 - active) * h
                c = active * c_new + (1.0 - active) * c
            else:
                h, c = h_new, c_new
            outputs.append(h)
        out = torch.stack(outputs, dim=1)
        return self.dropout(out), (h, c)


class UniLSTMEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, layers: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.ModuleList()
        current = input_dim
        for _ in range(layers):
            self.layers.append(UniLSTMLayer(current, hidden_dim, dropout))
            current = hidden_dim

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
        states: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
    ):
        new_states = []
        raw_outputs = []
        for i, layer in enumerate(self.layers):
            state = None if states is None else states[i]
            x, state = layer(x, lengths, state)
            raw_outputs.append(x)
            new_states.append(state)
        return x, new_states, raw_outputs


class ScratchLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int,
        hidden_dim: int,
        layers: int,
        dropout: float,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.encoder = UniLSTMEncoder(emb_dim, hidden_dim, layers, dropout)
        self.decoder = nn.Linear(hidden_dim, vocab_size)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.05)
        if emb_dim == hidden_dim:
            self.decoder.weight = self.embedding.weight

    def forward(self, x: torch.Tensor, states=None):
        emb = self.emb_dropout(self.embedding(x))
        out, states, raw_outputs = self.encoder(emb, None, states)
        logits = self.decoder(out)
        return logits, states, raw_outputs


class ULMFiTClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int,
        hidden_dim: int,
        layers: int,
        dropout: float,
        pad_idx: int = 0,
        unk_idx: int = 1,
        word_dropout: float = 0.03,
    ) -> None:
        super().__init__()
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.word_dropout = word_dropout
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.encoder = UniLSTMEncoder(emb_dim, hidden_dim, layers, dropout)
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        feature_dim = hidden_dim * 4
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def load_lm(self, lm: ScratchLanguageModel) -> None:
        self.embedding.load_state_dict(lm.embedding.state_dict())
        self.encoder.load_state_dict(lm.encoder.state_dict())

    def _word_dropout(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.word_dropout <= 0.0:
            return x
        mask = (
            torch.rand(x.shape, device=x.device) < self.word_dropout
        ) & (x != self.pad_idx)
        return x.masked_fill(mask, self.unk_idx)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        x = self._word_dropout(x)
        emb = self.emb_dropout(self.embedding(x))
        out, _, _ = self.encoder(emb, lengths, None)
        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        last_idx = (lengths - 1).clamp_min(0).view(-1, 1, 1).expand(-1, 1, out.size(2))
        last = out.gather(1, last_idx).squeeze(1)
        max_pool = out.masked_fill(~mask.unsqueeze(2), -1e4).max(dim=1).values
        mean_pool = (
            out.masked_fill(~mask.unsqueeze(2), 0.0).sum(dim=1)
            / lengths.clamp_min(1).to(out.dtype).unsqueeze(1)
        )
        attn_score = self.attn(out).squeeze(2).masked_fill(~mask, -1e4)
        attn = F.softmax(attn_score, dim=1).unsqueeze(2)
        attn_pool = (out * attn).sum(dim=1)
        return self.head(torch.cat([last, max_pool, mean_pool, attn_pool], dim=1)).squeeze(1)


def build_lm_stream(texts: Sequence[str], word2idx: dict, seq_len: int) -> List[int]:
    eos_id = word2idx[EOS]
    ids = []
    for text in texts:
        doc_ids, _ = encode(tokenize(text), word2idx, seq_len=10_000)
        while doc_ids and doc_ids[-1] == 0:
            doc_ids.pop()
        ids.extend(doc_ids)
        ids.append(eos_id)
    return ids


def batchify(ids: Sequence[int], batch_size: int, device: torch.device) -> torch.Tensor:
    n_batch = len(ids) // batch_size
    data = torch.tensor(ids[: n_batch * batch_size], dtype=torch.long, device=device)
    return data.view(batch_size, n_batch).contiguous()


def detach_states(states):
    if states is None:
        return None
    return [(h.detach(), c.detach()) for h, c in states]


def train_language_model(lm, data, args, device):
    optimizer = torch.optim.AdamW(lm.parameters(), lr=args.lm_lr, weight_decay=args.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    best_loss = float("inf")
    best_path = os.path.join(args.out_dir, "lm_best.pt")
    steps_per_epoch = max(1, (data.size(1) - 1) // args.bptt)
    total_steps = steps_per_epoch * args.lm_epochs
    warmup = max(1, int(total_steps * 0.05))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.2 + 0.8 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    global_step = 0
    for epoch in range(1, args.lm_epochs + 1):
        lm.train()
        states = None
        total_loss = 0.0
        total_tokens = 0
        positions = list(range(0, data.size(1) - 1, args.bptt))
        for step, i in enumerate(positions, 1):
            seq_len = min(args.bptt, data.size(1) - 1 - i)
            x = data[:, i : i + seq_len]
            y = data[:, i + 1 : i + 1 + seq_len]
            states = detach_states(states)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp and device.type == "cuda"):
                logits, states, raw_outputs = lm(x, states)
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
                if args.alpha:
                    loss = loss + args.alpha * raw_outputs[-1].pow(2).mean()
                if args.beta and raw_outputs[-1].size(1) > 1:
                    diff = raw_outputs[-1][:, 1:, :] - raw_outputs[-1][:, :-1, :]
                    loss = loss + args.beta * diff.pow(2).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(lm.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1
            total_loss += loss.item() * y.numel()
            total_tokens += y.numel()
            if step % args.log_every == 0:
                print(
                    f"lm epoch {epoch:02d} step {step}/{len(positions)} "
                    f"loss={total_loss / max(1,total_tokens):.4f} "
                    f"ppl={math.exp(min(20,total_loss / max(1,total_tokens))):.2f}"
                )
        epoch_loss = total_loss / max(1, total_tokens)
        print(f"lm epoch {epoch:02d} loss={epoch_loss:.4f} ppl={math.exp(min(20, epoch_loss)):.2f}")
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({"model": lm.state_dict(), "args": vars(args), "lm_loss": best_loss}, best_path)
            print(f"saved {best_path} lm_loss={best_loss:.4f}")
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    lm.load_state_dict(ckpt["model"])


def train_classifier(model, train_data: EncodedDataset, valid_data: EncodedDataset, args, device):
    train_batcher = BucketBatcher(train_data, args.batch_size, True, args.seed)
    valid_batcher = BucketBatcher(valid_data, args.batch_size, False, args.seed)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.embedding.parameters(), "lr": args.clf_lr * 0.25},
            {"params": model.encoder.parameters(), "lr": args.clf_lr * 0.5},
            {"params": model.head.parameters(), "lr": args.clf_lr},
            {"params": model.attn.parameters(), "lr": args.clf_lr},
        ],
        weight_decay=args.weight_decay,
    )
    total_steps = len(train_batcher) * args.clf_epochs
    warmup = max(1, int(total_steps * 0.08))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    best_acc = -1.0
    best_path = os.path.join(args.out_dir, "ulmfit_clf_best.pt")
    for epoch in range(1, args.clf_epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0
        for batch in train_batcher:
            x, lengths, labels = [item.to(device, non_blocking=True) for item in batch]
            targets = labels * (1.0 - 2.0 * args.label_smoothing) + args.label_smoothing
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp and device.type == "cuda"):
                logits = model(x, lengths)
                loss = F.binary_cross_entropy_with_logits(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total += labels.numel()
                total_loss += loss.item() * labels.numel()
        val_loss, val_acc = evaluate_classifier(model, valid_batcher, device)
        print(
            f"clf epoch {epoch:02d} train_loss={total_loss/max(1,total):.5f} "
            f"train_acc={total_correct/max(1,total):.4f} "
            f"val_loss={val_loss:.5f} val_acc={val_acc:.4f}"
        )
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({"model": model.state_dict(), "args": vars(args), "val_acc": best_acc}, best_path)
            print(f"saved {best_path} val_acc={best_acc:.4f}")
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    return best_acc


def evaluate_classifier(model, batcher, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in batcher:
            x, lengths, labels = [item.to(device, non_blocking=True) for item in batch]
            logits = model(x, lengths)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * labels.numel()
            total_correct += (pred == labels).sum().item()
            total += labels.numel()
    return total_loss / total, total_correct / total


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--unlabel", default="train_unlabel.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--out-dir", default="runs_ulmfit_scratch")
    parser.add_argument("--submission", default="submission_ulmfit.csv")
    parser.add_argument("--seq-len", type=int, default=512)
    parser.add_argument("--bptt", type=int, default=80)
    parser.add_argument("--min-count", type=int, default=3)
    parser.add_argument("--max-vocab", type=int, default=60000)
    parser.add_argument("--emb-dim", type=int, default=256)
    parser.add_argument("--hidden-dim", type=int, default=256)
    parser.add_argument("--layers", type=int, default=2)
    parser.add_argument("--dropout", type=float, default=0.35)
    parser.add_argument("--word-dropout", type=float, default=0.04)
    parser.add_argument("--lm-batch-size", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=96)
    parser.add_argument("--lm-epochs", type=int, default=2)
    parser.add_argument("--clf-epochs", type=int, default=6)
    parser.add_argument("--lm-lr", type=float, default=0.0015)
    parser.add_argument("--clf-lr", type=float, default=0.001)
    parser.add_argument("--weight-decay", type=float, default=0.0002)
    parser.add_argument("--alpha", type=float, default=1e-4)
    parser.add_argument("--beta", type=float, default=1e-4)
    parser.add_argument("--grad-clip", type=float, default=0.25)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2029)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-train", type=int, default=None)
    parser.add_argument("--max-unlabel", type=int, default=None)
    parser.add_argument("--max-lm-docs", type=int, default=None)
    parser.add_argument("--log-every", type=int, default=300)
    parser.add_argument("--skip-lm", action="store_true")
    parser.add_argument("--lm-checkpoint", default=None)
    parser.add_argument("--skip-predict", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")
    train_texts, train_labels = read_train(args.train, args.max_train)
    unlabel_texts = read_unlabel(args.unlabel, args.max_unlabel)
    test_ids, test_texts = read_test(args.test)
    print(f"loaded train={len(train_texts)} unlabel={len(unlabel_texts)} test={len(test_texts)}")

    vocab_docs = train_texts + unlabel_texts
    tokenized = [tokenize(text) for text in vocab_docs]
    word2idx, idx2word = build_vocab(tokenized + [[EOS]], args.min_count, args.max_vocab)
    if EOS not in word2idx:
        word2idx[EOS] = len(idx2word)
        idx2word.append(EOS)
    print(f"vocab={len(idx2word)}")

    lm_docs = train_texts + unlabel_texts
    random.Random(args.seed).shuffle(lm_docs)
    if args.max_lm_docs:
        lm_docs = lm_docs[: args.max_lm_docs]
    print("building LM stream ...")
    lm_ids = build_lm_stream(lm_docs, word2idx, args.seq_len)
    lm_data = batchify(lm_ids, args.lm_batch_size, device)
    print(f"lm_tokens={len(lm_ids)} batches_per_epoch={(lm_data.size(1)-1)//args.bptt}")

    lm = ScratchLanguageModel(
        len(idx2word), args.emb_dim, args.hidden_dim, args.layers, args.dropout
    ).to(device)
    if args.lm_checkpoint:
        ckpt = torch.load(args.lm_checkpoint, map_location=device, weights_only=False)
        lm.load_state_dict(ckpt["model"])
    if not args.skip_lm:
        print("pretraining scratch language model ...")
        train_language_model(lm, lm_data, args, device)

    full_train = make_dataset(train_texts, train_labels, word2idx, args.seq_len)
    train_idx, valid_idx = split_indices(len(full_train), args.valid_ratio, args.seed)
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)
    classifier = ULMFiTClassifier(
        len(idx2word),
        args.emb_dim,
        args.hidden_dim,
        args.layers,
        args.dropout,
        word_dropout=args.word_dropout,
    ).to(device)
    classifier.load_lm(lm)
    print("fine-tuning classifier from LM ...")
    best_acc = train_classifier(classifier, train_data, valid_data, args, device)
    print(f"best ulmfit val_acc={best_acc:.4f}")

    if not args.skip_predict:
        test_data = make_dataset(test_texts, None, word2idx, args.seq_len)
        probs = predict_probs(classifier, test_data, args.batch_size, device)
        write_submission(test_ids, probs, args.submission)
        print(f"wrote {args.submission}")
    torch.save(
        {
            "model": classifier.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": vars(args),
            "val_acc": best_acc,
            "model_class": "ULMFiTClassifier",
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第四部分：强LSTM模型（来自train_strong_lstm.py）
# ============================================
# 功能：实现增强的LSTM分类器
# 特性：多层BiLSTM、CNN多尺度特征、嵌入层通道dropout
# 使用方法：运行此单元格后，可以使用StrongTextLSTMClassifier类

import argparse
import math
import os
from argparse import Namespace
from typing import List

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    ScratchBiLSTM,
    build_vocab,
    evaluate,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    set_seed,
    split_indices,
    subset_dataset,
    tokenize,
    train_model,
    write_submission,
)


class EmbeddingChannelDropout(nn.Module):
    def __init__(self, p: float) -> None:
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p <= 0.0:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty(x.size(0), 1, x.size(2)).bernoulli_(keep) / keep
        return x * mask


class StackedScratchBiLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, layers: int, dropout: float):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        current_dim = input_dim
        for _ in range(layers):
            self.layers.append(ScratchBiLSTM(current_dim, hidden_dim, dropout))
            self.norms.append(nn.LayerNorm(hidden_dim * 2))
            current_dim = hidden_dim * 2
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor):
        last = None
        for i, layer in enumerate(self.layers):
            residual = x
            x, last = layer(x, lengths)
            x = self.norms[i](x)
            if residual.shape == x.shape:
                x = x + residual
            x = self.dropout(x)
        return x, last


class StrongTextLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int,
        hidden_dim: int,
        dropout: float,
        layers: int,
        conv_channels: int,
        word_dropout: float,
        emb_channel_dropout: float,
        pad_idx: int = 0,
        unk_idx: int = 1,
    ) -> None:
        super().__init__()
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.word_dropout = word_dropout
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.emb_channel_dropout = EmbeddingChannelDropout(emb_channel_dropout)
        self.encoder = StackedScratchBiLSTM(emb_dim, hidden_dim, layers, dropout)
        out_dim = hidden_dim * 2
        self.attn = nn.Sequential(
            nn.Linear(out_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        kernels = [2, 3, 4, 5]
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(emb_dim, conv_channels, kernel_size=k, padding=0, bias=False)
                for k in kernels
            ]
        )
        feature_dim = out_dim * 4 + conv_channels * len(kernels)
        self.fc = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.05)
        with torch.no_grad():
            self.embedding.weight[pad_idx].zero_()

    def _word_dropout(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.word_dropout <= 0.0:
            return x
        mask = (
            torch.rand(x.shape, device=x.device) < self.word_dropout
        ) & (x != self.pad_idx)
        return x.masked_fill(mask, self.unk_idx)

    def _conv_features(self, emb: torch.Tensor) -> torch.Tensor:
        conv_in = emb.transpose(1, 2)
        pooled: List[torch.Tensor] = []
        for conv in self.convs:
            feat = F.gelu(conv(conv_in))
            pooled.append(F.adaptive_max_pool1d(feat, 1).squeeze(2))
        return torch.cat(pooled, dim=1)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        x = self._word_dropout(x)
        emb = self.embedding(x)
        emb = self.emb_channel_dropout(self.emb_dropout(emb))
        conv_pool = self._conv_features(emb)
        out, last = self.encoder(emb, lengths)

        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        attn_score = self.attn(out).squeeze(2).masked_fill(~mask, -1e4)
        attn_weight = F.softmax(attn_score, dim=1).unsqueeze(2)
        attn_pool = torch.sum(out * attn_weight, dim=1)
        max_pool = out.masked_fill(~mask.unsqueeze(2), -1e4).max(dim=1).values
        mean_pool = (
            out.masked_fill(~mask.unsqueeze(2), 0.0).sum(dim=1)
            / lengths.clamp_min(1).to(out.dtype).unsqueeze(1)
        )
        features = torch.cat([last, attn_pool, max_pool, mean_pool, conv_pool], dim=1)
        return self.fc(features).squeeze(1)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--unlabel", default="train_unlabel.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--out-dir", default="runs_strong_lstm")
    parser.add_argument("--submission", default="submission_strong.csv")
    parser.add_argument("--seq-len", type=int, default=512)
    parser.add_argument("--min-count", type=int, default=2)
    parser.add_argument("--max-vocab", type=int, default=100000)
    parser.add_argument("--emb-dim", type=int, default=256)
    parser.add_argument("--hidden-dim", type=int, default=192)
    parser.add_argument("--layers", type=int, default=2)
    parser.add_argument("--conv-channels", type=int, default=96)
    parser.add_argument("--dropout", type=float, default=0.42)
    parser.add_argument("--word-dropout", type=float, default=0.04)
    parser.add_argument("--emb-channel-dropout", type=float, default=0.18)
    parser.add_argument("--batch-size", type=int, default=96)
    parser.add_argument("--epochs", type=int, default=7)
    parser.add_argument("--lr", type=float, default=0.0012)
    parser.add_argument("--weight-decay", type=float, default=0.0003)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--warmup-ratio", type=float, default=0.08)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2027)
    parser.add_argument("--include-test-vocab", action="store_true", default=True)
    parser.add_argument("--no-include-test-vocab", dest="include_test_vocab", action="store_false")
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-train", type=int, default=None)
    parser.add_argument("--max-unlabel", type=int, default=None)
    parser.add_argument("--skip-predict", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")

    train_texts, train_labels = read_train(args.train, args.max_train)
    unlabel_texts = read_unlabel(args.unlabel, args.max_unlabel)
    test_ids, test_texts = read_test(args.test)
    print(f"loaded train={len(train_texts)} unlabel={len(unlabel_texts)} test={len(test_texts)}")

    vocab_texts = train_texts + unlabel_texts
    if args.include_test_vocab:
        vocab_texts += test_texts
    print("tokenizing for vocabulary ...")
    word2idx, idx2word = build_vocab(
        [tokenize(text) for text in vocab_texts], args.min_count, args.max_vocab
    )
    print(f"vocab={len(idx2word)}")

    full_train = make_dataset(train_texts, train_labels, word2idx, args.seq_len)
    train_idx, valid_idx = split_indices(len(full_train), args.valid_ratio, args.seed)
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)

    model = StrongTextLSTMClassifier(
        vocab_size=len(idx2word),
        emb_dim=args.emb_dim,
        hidden_dim=args.hidden_dim,
        dropout=args.dropout,
        layers=args.layers,
        conv_channels=args.conv_channels,
        word_dropout=args.word_dropout,
        emb_channel_dropout=args.emb_channel_dropout,
    ).to(device)
    print("training strong LSTM-like model ...")
    best_acc = train_model(model, train_data, valid_data, args, device, "strong")
    print(f"best strong val_acc={best_acc:.4f}")

    if not args.skip_predict:
        test_data = make_dataset(test_texts, None, word2idx, args.seq_len)
        probs = predict_probs(model, test_data, args.batch_size, device)
        write_submission(test_ids, probs, args.submission)
        print(f"wrote {args.submission}")

    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": vars(args),
            "val_acc": best_acc,
            "model_class": "StrongTextLSTMClassifier",
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第五部分：ELMo风格模型（来自train_elmo_scratch.py）
# ============================================
# 功能：实现类似ELMo的双向语言模型融合分类器
# 特性：前向+后向LM预训练、多层特征混合、多种池化策略
# 使用方法：需要先运行ULMFiT部分获取预训练权重

import argparse
import csv
import math
import os
import random
from typing import List, Sequence

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    BucketBatcher,
    EncodedDataset,
    encode,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    set_seed,
    split_indices,
    subset_dataset,
    tokenize,
    write_submission,
)
from train_ulmfit_scratch import (
    EOS,
    ScratchLanguageModel,
    UniLSTMEncoder,
    batchify,
    train_language_model,
)


def build_reverse_lm_stream(texts: Sequence[str], word2idx: dict, seq_len: int) -> List[int]:
    eos_id = word2idx[EOS]
    ids: List[int] = []
    for text in texts:
        tokens = tokenize(text)
        if len(tokens) > seq_len:
            head = seq_len // 2
            tail = seq_len - head
            tokens = tokens[:head] + tokens[-tail:]
        doc_ids = [word2idx.get(token, 1) for token in reversed(tokens)]
        ids.extend(doc_ids)
        ids.append(eos_id)
    return ids


class ELMoStyleClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int,
        hidden_dim: int,
        layers: int,
        dropout: float,
        word_dropout: float,
        pad_idx: int = 0,
        unk_idx: int = 1,
    ) -> None:
        super().__init__()
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.word_dropout = word_dropout
        self.fwd_embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.bwd_embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.fwd_encoder = UniLSTMEncoder(emb_dim, hidden_dim, layers, dropout)
        self.bwd_encoder = UniLSTMEncoder(emb_dim, hidden_dim, layers, dropout)
        self.emb_dropout = nn.Dropout(dropout)
        out_dim = hidden_dim * 2
        self.layer_mix = nn.Parameter(torch.zeros(layers))
        self.attn = nn.Sequential(
            nn.Linear(out_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.conv3 = nn.Conv1d(out_dim, hidden_dim, kernel_size=3, padding=1, bias=False)
        feature_dim = out_dim * 4 + hidden_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, 1),
        )

    def load_lms(self, fwd_lm: ScratchLanguageModel, bwd_lm: ScratchLanguageModel) -> None:
        self.fwd_embedding.load_state_dict(fwd_lm.embedding.state_dict())
        self.bwd_embedding.load_state_dict(bwd_lm.embedding.state_dict())
        self.fwd_encoder.load_state_dict(fwd_lm.encoder.state_dict())
        self.bwd_encoder.load_state_dict(bwd_lm.encoder.state_dict())

    def _word_dropout(self, x):
        if not self.training or self.word_dropout <= 0:
            return x
        mask = (torch.rand(x.shape, device=x.device) < self.word_dropout) & (x != self.pad_idx)
        return x.masked_fill(mask, self.unk_idx)

    def _mix_layers(self, fwd_raw, bwd_raw):
        weights = F.softmax(self.layer_mix, dim=0)
        mixed = None
        for i, weight in enumerate(weights):
            bwd = torch.flip(bwd_raw[i], dims=[1])
            layer = torch.cat([fwd_raw[i], bwd], dim=2)
            mixed = layer * weight if mixed is None else mixed + layer * weight
        return mixed

    def forward(self, x, lengths):
        x = self._word_dropout(x)
        fwd_emb = self.emb_dropout(self.fwd_embedding(x))
        fwd_out, _, fwd_raw = self.fwd_encoder(fwd_emb, lengths, None)

        bwd_x = torch.flip(x, dims=[1])
        bwd_lengths = lengths
        bwd_emb = self.emb_dropout(self.bwd_embedding(bwd_x))
        bwd_out, _, bwd_raw = self.bwd_encoder(bwd_emb, bwd_lengths, None)
        out = self._mix_layers(fwd_raw, bwd_raw)

        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        last_idx = (lengths - 1).clamp_min(0).view(-1, 1, 1).expand(-1, 1, out.size(2))
        last = out.gather(1, last_idx).squeeze(1)
        first = out[:, 0, :]
        max_pool = out.masked_fill(~mask.unsqueeze(2), -1e4).max(dim=1).values
        mean_pool = (
            out.masked_fill(~mask.unsqueeze(2), 0.0).sum(dim=1)
            / lengths.clamp_min(1).to(out.dtype).unsqueeze(1)
        )
        attn_score = self.attn(out).squeeze(2).masked_fill(~mask, -1e4)
        attn = F.softmax(attn_score, dim=1).unsqueeze(2)
        attn_pool = (out * attn).sum(dim=1)
        conv_pool = F.adaptive_max_pool1d(F.gelu(self.conv3(out.transpose(1, 2))), 1).squeeze(2)
        return self.head(torch.cat([first + last, max_pool, mean_pool, attn_pool, conv_pool], dim=1)).squeeze(1)


def evaluate(model, batcher, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in batcher:
            x, lengths, labels = [item.to(device, non_blocking=True) for item in batch]
            logits = model(x, lengths)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * labels.numel()
            total_correct += (pred == labels).sum().item()
            total += labels.numel()
    return total_loss / total, total_correct / total


def train_classifier(model, train_data: EncodedDataset, valid_data: EncodedDataset, args, device):
    train_batcher = BucketBatcher(train_data, args.batch_size, True, args.seed)
    valid_batcher = BucketBatcher(valid_data, args.batch_size, False, args.seed)
    optimizer = torch.optim.AdamW(
        [
            {"params": list(model.fwd_embedding.parameters()) + list(model.bwd_embedding.parameters()), "lr": args.clf_lr * 0.2},
            {"params": list(model.fwd_encoder.parameters()) + list(model.bwd_encoder.parameters()), "lr": args.clf_lr * 0.45},
            {"params": list(model.head.parameters()) + list(model.attn.parameters()) + list(model.conv3.parameters()) + [model.layer_mix], "lr": args.clf_lr},
        ],
        weight_decay=args.weight_decay,
    )
    total_steps = len(train_batcher) * args.clf_epochs
    warmup = max(1, int(total_steps * 0.08))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    best_acc = -1.0
    best_path = os.path.join(args.out_dir, "elmo_clf_best.pt")
    for epoch in range(1, args.clf_epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0
        for batch in train_batcher:
            x, lengths, labels = [item.to(device, non_blocking=True) for item in batch]
            targets = labels * (1 - 2 * args.label_smoothing) + args.label_smoothing
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp and device.type == "cuda"):
                logits = model(x, lengths)
                loss = F.binary_cross_entropy_with_logits(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total += labels.numel()
                total_loss += loss.item() * labels.numel()
        val_loss, val_acc = evaluate(model, valid_batcher, device)
        print(
            f"elmo clf epoch {epoch:02d} train_loss={total_loss/max(1,total):.5f} "
            f"train_acc={total_correct/max(1,total):.4f} val_loss={val_loss:.5f} val_acc={val_acc:.4f}"
        )
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({"model": model.state_dict(), "args": vars(args), "val_acc": best_acc}, best_path)
            print(f"saved {best_path} val_acc={best_acc:.4f}")
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    return best_acc


def calibrate_and_write(model, valid_data, test_ids, test_texts, word2idx, args, device):
    valid_probs = predict_probs(model, valid_data, args.batch_size, device)
    labels = valid_data.y.tolist()
    best = (0.0, 0.5)
    for i in range(1, 1000):
        th = i / 1000
        acc = sum((p >= th) == (y >= 0.5) for p, y in zip(valid_probs, labels)) / len(labels)
        if acc > best[0]:
            best = (acc, th)
    test_data = make_dataset(test_texts, None, word2idx, args.seq_len)
    test_probs = predict_probs(model, test_data, args.batch_size, device)
    write_submission(test_ids, test_probs, args.submission)
    with open(args.submission.replace(".csv", "_calibrated.csv"), "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, prob in zip(test_ids, test_probs):
            writer.writerow([sample_id, int(prob >= best[1])])
    order = sorted(range(len(test_probs)), key=lambda i: test_probs[i], reverse=True)
    labels_balanced = [0] * len(test_probs)
    for i in order[: len(test_probs) // 2]:
        labels_balanced[i] = 1
    with open(args.submission.replace(".csv", "_balanced.csv"), "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, label in zip(test_ids, labels_balanced):
            writer.writerow([sample_id, label])
    print(f"calibrated_val_acc={best[0]:.4f} threshold={best[1]:.3f}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-final", default="runs_ulmfit_full/final_model.pt")
    parser.add_argument("--fwd-lm", default="runs_ulmfit_full/lm_best.pt")
    parser.add_argument("--out-dir", default="runs_elmo_scratch")
    parser.add_argument("--submission", default="submission_elmo.csv")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--unlabel", default="train_unlabel.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--seq-len", type=int, default=512)
    parser.add_argument("--bptt", type=int, default=80)
    parser.add_argument("--lm-batch-size", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--bwd-lm-epochs", type=int, default=2)
    parser.add_argument("--clf-epochs", type=int, default=5)
    parser.add_argument("--bwd-lm-lr", type=float, default=0.0015)
    parser.add_argument("--clf-lr", type=float, default=0.0008)
    parser.add_argument("--dropout", type=float, default=0.38)
    parser.add_argument("--word-dropout", type=float, default=0.04)
    parser.add_argument("--weight-decay", type=float, default=0.00025)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--grad-clip", type=float, default=0.25)
    parser.add_argument("--alpha", type=float, default=1e-4)
    parser.add_argument("--beta", type=float, default=1e-4)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2031)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-lm-docs", type=int, default=None)
    parser.add_argument("--max-train", type=int, default=None)
    parser.add_argument("--max-unlabel", type=int, default=None)
    parser.add_argument("--log-every", type=int, default=300)
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base = torch.load(args.base_final, map_location="cpu", weights_only=False)
    base_args = base["args"]
    word2idx = base["word2idx"]
    idx2word = base["idx2word"]
    emb_dim = base_args["emb_dim"]
    hidden_dim = base_args["hidden_dim"]
    layers = base_args["layers"]
    print(f"device={device} vocab={len(idx2word)} emb={emb_dim} hidden={hidden_dim} layers={layers}")

    train_texts, train_labels = read_train(args.train, args.max_train)
    unlabel_texts = read_unlabel(args.unlabel, args.max_unlabel)
    test_ids, test_texts = read_test(args.test)

    fwd_lm = ScratchLanguageModel(len(idx2word), emb_dim, hidden_dim, layers, base_args.get("dropout", 0.35)).to(device)
    fwd_ckpt = torch.load(args.fwd_lm, map_location=device, weights_only=False)
    fwd_lm.load_state_dict(fwd_ckpt["model"])

    bwd_lm = ScratchLanguageModel(len(idx2word), emb_dim, hidden_dim, layers, base_args.get("dropout", 0.35)).to(device)
    lm_docs = train_texts + unlabel_texts
    random.Random(args.seed).shuffle(lm_docs)
    if args.max_lm_docs:
        lm_docs = lm_docs[: args.max_lm_docs]
    print("building backward LM stream ...")
    bwd_ids = build_reverse_lm_stream(lm_docs, word2idx, args.seq_len)
    bwd_data = batchify(bwd_ids, args.lm_batch_size, device)
    print(f"bwd_lm_tokens={len(bwd_ids)} batches_per_epoch={(bwd_data.size(1)-1)//args.bptt}")
    lm_args = argparse.Namespace(**vars(args))
    lm_args.lm_lr = args.bwd_lm_lr
    lm_args.lm_epochs = args.bwd_lm_epochs
    print("pretraining backward LM ...")
    train_language_model(bwd_lm, bwd_data, lm_args, device)

    full_train = make_dataset(train_texts, train_labels, word2idx, args.seq_len)
    train_idx, valid_idx = split_indices(len(full_train), args.valid_ratio, args.seed)
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)
    classifier = ELMoStyleClassifier(
        len(idx2word), emb_dim, hidden_dim, layers, args.dropout, args.word_dropout
    ).to(device)
    classifier.load_lms(fwd_lm, bwd_lm)
    print("training ELMo-style classifier ...")
    best_acc = train_classifier(classifier, train_data, valid_data, args, device)
    print(f"best_elmo_val_acc={best_acc:.4f}")
    calibrate_and_write(classifier, valid_data, test_ids, test_texts, word2idx, args, device)
    torch.save(
        {
            "model": classifier.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": vars(args),
            "val_acc": best_acc,
            "model_class": "ELMoStyleClassifier",
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第六部分：n-gram混合模型（来自train_ulmfit_ngram.py）
# ============================================
# 功能：结合ULMFiT和n-gram特征的混合模型
# 特性：哈希n-gram特征、可学习的特征混合权重
# 使用方法：需要提供ULMFiT预训练的检查点

import argparse
import csv
import math
import os
import random
from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    BucketBatcher,
    EncodedDataset,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    split_indices,
    subset_dataset,
    tokenize,
)
from train_ulmfit_scratch import ULMFiTClassifier, evaluate_classifier


HASH_BASE = 1000003


def stable_hash_ids(parts: Sequence[int], buckets: int) -> int:
    h = 1469598103934665603
    for part in parts:
        h ^= part + 0x9E3779B97F4A7C15
        h = (h * HASH_BASE) & 0xFFFFFFFFFFFFFFFF
    return 1 + (h % (buckets - 1))


def ngram_ids_for_text(
    text: str,
    word2idx: dict,
    buckets: int,
    max_ngrams: int,
    max_tokens: int,
    orders: Tuple[int, ...],
) -> List[int]:
    token_ids = [word2idx.get(token, 1) for token in tokenize(text)[:max_tokens]]
    ids = set()
    for order in orders:
        if len(token_ids) < order:
            continue
        for i in range(len(token_ids) - order + 1):
            ids.add(stable_hash_ids(token_ids[i : i + order], buckets))
    ids = sorted(ids)
    if len(ids) > max_ngrams:
        # Keep a deterministic spread over the full document instead of only the beginning.
        step = len(ids) / max_ngrams
        ids = [ids[int(i * step)] for i in range(max_ngrams)]
    return ids


@dataclass
class HybridDataset:
    seq: EncodedDataset
    ngrams: torch.Tensor
    ngram_counts: torch.Tensor

    def __len__(self):
        return len(self.seq)


class HybridBatcher:
    def __init__(self, dataset: HybridDataset, batch_size: int, shuffle: bool, seed: int):
        self.seq_batcher = BucketBatcher(dataset.seq, batch_size, shuffle, seed)
        self.dataset = dataset

    def __len__(self):
        return len(self.seq_batcher)

    def __iter__(self):
        for batch in self.seq_batcher:
            x = batch[0]
            # Recover selected rows by matching sequence rows is unsafe; use a direct local batcher instead.
            raise RuntimeError("HybridBatcher should not be used")


class HybridIndexBatcher:
    def __init__(self, dataset: HybridDataset, batch_size: int, shuffle: bool, seed: int, bucket_size: int = 2048):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.bucket_size = bucket_size
        self.epoch = 0

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1
        if self.shuffle:
            rng.shuffle(indices)
            buckets = [indices[i : i + self.bucket_size] for i in range(0, len(indices), self.bucket_size)]
            for bucket in buckets:
                bucket.sort(key=lambda idx: int(self.dataset.seq.lengths[idx]), reverse=True)
            rng.shuffle(buckets)
            indices = [idx for bucket in buckets for idx in bucket]
        else:
            indices.sort(key=lambda idx: int(self.dataset.seq.lengths[idx]), reverse=True)

        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start : start + self.batch_size]
            idx = torch.tensor(batch_idx, dtype=torch.long)
            fields = [
                self.dataset.seq.x.index_select(0, idx),
                self.dataset.seq.lengths.index_select(0, idx),
                self.dataset.ngrams.index_select(0, idx),
                self.dataset.ngram_counts.index_select(0, idx),
            ]
            if self.dataset.seq.y is not None:
                fields.append(self.dataset.seq.y.index_select(0, idx))
            yield tuple(fields)


def make_hybrid_dataset(
    texts: Sequence[str],
    labels: Optional[Sequence[float]],
    word2idx: dict,
    seq_len: int,
    buckets: int,
    max_ngrams: int,
    max_ngram_tokens: int,
    orders: Tuple[int, ...],
) -> HybridDataset:
    seq = make_dataset(texts, labels, word2idx, seq_len)
    rows = []
    counts = []
    for text in texts:
        ids = ngram_ids_for_text(
            text, word2idx, buckets, max_ngrams, max_ngram_tokens, orders
        )
        counts.append(max(1, len(ids)))
        if len(ids) < max_ngrams:
            ids = ids + [0] * (max_ngrams - len(ids))
        rows.append(ids)
    return HybridDataset(
        seq=seq,
        ngrams=torch.tensor(rows, dtype=torch.long),
        ngram_counts=torch.tensor(counts, dtype=torch.float32),
    )


def subset_hybrid(dataset: HybridDataset, indices: Sequence[int]) -> HybridDataset:
    idx = torch.tensor(indices, dtype=torch.long)
    return HybridDataset(
        seq=subset_dataset(dataset.seq, indices),
        ngrams=dataset.ngrams.index_select(0, idx),
        ngram_counts=dataset.ngram_counts.index_select(0, idx),
    )


class ULMFiTNgramClassifier(nn.Module):
    def __init__(
        self,
        base: ULMFiTClassifier,
        buckets: int,
        ngram_dropout: float,
        ngram_scale: float,
    ):
        super().__init__()
        self.base = base
        self.ngram_weight = nn.Embedding(buckets, 1, padding_idx=0)
        self.ngram_bias = nn.Parameter(torch.zeros(1))
        self.mix = nn.Parameter(torch.tensor([1.0, ngram_scale]))
        self.ngram_dropout = ngram_dropout
        nn.init.zeros_(self.ngram_weight.weight)

    def forward(self, x, lengths, ngrams, ngram_counts):
        base_logit = self.base(x, lengths)
        weights = self.ngram_weight(ngrams).squeeze(2)
        mask = (ngrams != 0).to(weights.dtype)
        if self.training and self.ngram_dropout > 0:
            keep = (torch.rand_like(mask) > self.ngram_dropout).to(weights.dtype)
            mask = mask * keep
        ngram_logit = (weights * mask).sum(dim=1) / ngram_counts.sqrt().to(weights.dtype).clamp_min(1.0)
        ngram_logit = ngram_logit + self.ngram_bias
        return self.mix[0] * base_logit + self.mix[1] * ngram_logit


def evaluate(model, batcher, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in batcher:
            x, lengths, ngrams, counts, labels = [item.to(device, non_blocking=True) for item in batch]
            logits = model(x, lengths, ngrams, counts)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * labels.numel()
            total_correct += (pred == labels).sum().item()
            total += labels.numel()
    return total_loss / total, total_correct / total


def collect_probs(model, dataset: HybridDataset, batch_size: int, device):
    batcher = HybridIndexBatcher(dataset, batch_size, False, 1234)
    model.eval()
    indexed = []
    # Reimplement without losing order.
    indices = list(range(len(dataset)))
    indices.sort(key=lambda idx: int(dataset.seq.lengths[idx]), reverse=True)
    probs = [None] * len(dataset)
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start : start + batch_size]
            idx = torch.tensor(batch_idx, dtype=torch.long)
            x = dataset.seq.x.index_select(0, idx).to(device)
            lengths = dataset.seq.lengths.index_select(0, idx).to(device)
            ngrams = dataset.ngrams.index_select(0, idx).to(device)
            counts = dataset.ngram_counts.index_select(0, idx).to(device)
            batch_probs = torch.sigmoid(model(x, lengths, ngrams, counts)).cpu().tolist()
            for original, prob in zip(batch_idx, batch_probs):
                probs[original] = prob
    return [float(p) for p in probs]


def calibrate(model, valid_data, batch_size, device):
    probs = collect_probs(model, valid_data, batch_size, device)
    labels = valid_data.seq.y.tolist()
    best = (0.0, 0.5)
    for i in range(1, 1000):
        th = i / 1000
        acc = sum((p >= th) == (y >= 0.5) for p, y in zip(probs, labels)) / len(labels)
        if acc > best[0]:
            best = (acc, th)
    return best


def write_submissions(model, test_ids, test_data, args, device):
    probs = collect_probs(model, test_data, args.batch_size, device)
    for suffix, labels in [
        ("", [int(p >= 0.5) for p in probs]),
        ("_balanced", None),
    ]:
        path = args.submission.replace(".csv", f"{suffix}.csv")
        if labels is None:
            order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
            labels = [0] * len(probs)
            for i in order[: len(probs) // 2]:
                labels[i] = 1
        with open(path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "label"])
            for sample_id, label in zip(test_ids, labels):
                writer.writerow([sample_id, label])
        print(f"wrote {path}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="runs_ulmfit_continue/ulmfit_clf_best.pt")
    parser.add_argument("--base-final", default="runs_ulmfit_full/final_model.pt")
    parser.add_argument("--out-dir", default="runs_ulmfit_ngram")
    parser.add_argument("--submission", default="submission_ulmfit_ngram.csv")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--epochs", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--base-lr-mult", type=float, default=0.25)
    parser.add_argument("--weight-decay", type=float, default=3e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.02)
    parser.add_argument("--grad-clip", type=float, default=0.5)
    parser.add_argument("--buckets", type=int, default=1048576)
    parser.add_argument("--max-ngrams", type=int, default=900)
    parser.add_argument("--max-ngram-tokens", type=int, default=1400)
    parser.add_argument("--orders", default="1,2,3")
    parser.add_argument("--ngram-dropout", type=float, default=0.08)
    parser.add_argument("--ngram-scale", type=float, default=1.0)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2035)
    parser.add_argument("--split-seed", type=int, default=None)
    parser.add_argument("--pilot-train-size", type=int, default=None)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-train", type=int, default=None)
    parser.add_argument("--skip-predict", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    random.seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    base_final = torch.load(args.base_final, map_location="cpu", weights_only=False)
    cfg = dict(checkpoint["args"])
    word2idx = base_final["word2idx"]
    idx2word = base_final["idx2word"]
    train_texts, train_labels = read_train(args.train, args.max_train)
    test_ids, test_texts = read_test(args.test)
    orders = tuple(int(x) for x in args.orders.split(",") if x)
    print("building hybrid datasets ...")
    split_seed = args.split_seed if args.split_seed is not None else cfg.get("seed", args.seed)
    train_idx, valid_idx = split_indices(len(train_texts), args.valid_ratio, split_seed)
    if args.pilot_train_size is not None:
        rng = random.Random(args.seed)
        train_idx = list(train_idx)
        rng.shuffle(train_idx)
        train_idx = sorted(train_idx[: args.pilot_train_size])
        pilot_train_texts = [train_texts[i] for i in train_idx]
        pilot_train_labels = [train_labels[i] for i in train_idx]
        valid_texts = [train_texts[i] for i in valid_idx]
        valid_labels = [train_labels[i] for i in valid_idx]
        train_data = make_hybrid_dataset(
            pilot_train_texts,
            pilot_train_labels,
            word2idx,
            cfg["seq_len"],
            args.buckets,
            args.max_ngrams,
            args.max_ngram_tokens,
            orders,
        )
        valid_data = make_hybrid_dataset(
            valid_texts,
            valid_labels,
            word2idx,
            cfg["seq_len"],
            args.buckets,
            args.max_ngrams,
            args.max_ngram_tokens,
            orders,
        )
        print(
            f"clean pilot train={len(train_data)} valid={len(valid_data)} split_seed={split_seed}"
        )
    else:
        full = make_hybrid_dataset(
            train_texts,
            train_labels,
            word2idx,
            cfg["seq_len"],
            args.buckets,
            args.max_ngrams,
            args.max_ngram_tokens,
            orders,
        )
        train_data = subset_hybrid(full, train_idx)
        valid_data = subset_hybrid(full, valid_idx)
        print(f"full split train={len(train_data)} valid={len(valid_data)} split_seed={split_seed}")
    test_data = None
    if not args.skip_predict:
        test_data = make_hybrid_dataset(
            test_texts,
            None,
            word2idx,
            cfg["seq_len"],
            args.buckets,
            args.max_ngrams,
            args.max_ngram_tokens,
            orders,
        )

    base_model = ULMFiTClassifier(
        len(idx2word),
        cfg["emb_dim"],
        cfg["hidden_dim"],
        cfg["layers"],
        cfg["dropout"],
        word_dropout=cfg.get("word_dropout", 0.04),
    )
    base_model.load_state_dict(checkpoint["model"])
    model = ULMFiTNgramClassifier(base_model, args.buckets, args.ngram_dropout, args.ngram_scale).to(device)

    optimizer = torch.optim.AdamW(
        [
            {"params": model.base.parameters(), "lr": args.lr * args.base_lr_mult},
            {"params": model.ngram_weight.parameters(), "lr": args.lr},
            {"params": [model.ngram_bias, model.mix], "lr": args.lr},
        ],
        weight_decay=args.weight_decay,
    )
    train_batcher = HybridIndexBatcher(train_data, args.batch_size, True, args.seed)
    total_steps = len(train_batcher) * args.epochs
    warmup = max(1, int(total_steps * 0.08))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    best_acc = -1.0
    best_path = os.path.join(args.out_dir, "ulmfit_ngram_best.pt")

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0
        for batch in train_batcher:
            x, lengths, ngrams, counts, labels = [item.to(device, non_blocking=True) for item in batch]
            targets = labels * (1 - 2 * args.label_smoothing) + args.label_smoothing
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp and device.type == "cuda"):
                logits = model(x, lengths, ngrams, counts)
                loss = F.binary_cross_entropy_with_logits(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total += labels.numel()
                total_loss += loss.item() * labels.numel()
        valid_batcher = HybridIndexBatcher(valid_data, args.batch_size, False, args.seed)
        val_loss, val_acc = evaluate(model, valid_batcher, device)
        cal_acc, cal_th = calibrate(model, valid_data, args.batch_size, device)
        print(
            f"ngram epoch {epoch:02d} train_loss={total_loss/max(1,total):.5f} "
            f"train_acc={total_correct/max(1,total):.4f} val_acc={val_acc:.4f} "
            f"cal_acc={cal_acc:.4f} cal_th={cal_th:.3f} mix={model.mix.detach().cpu().tolist()}"
        )
        if cal_acc > best_acc:
            best_acc = cal_acc
            torch.save(
                {
                    "model": model.state_dict(),
                    "args": vars(args),
                    "base_cfg": cfg,
                    "word2idx": word2idx,
                    "idx2word": idx2word,
                    "calibrated_val_acc": best_acc,
                    "val_acc": val_acc,
                },
                best_path,
            )
            print(f"saved {best_path} cal_acc={best_acc:.4f}")

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    if not args.skip_predict:
        write_submissions(model, test_ids, test_data, args, device)
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": vars(args),
            "base_cfg": cfg,
            "calibrated_val_acc": best_acc,
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第七部分：VAT半监督模型（来自train_ulmfit_vat.py）
# ============================================
# 功能：虚拟对抗训练（Virtual Adversarial Training）
# 特性：利用无标签数据进行半监督学习、对抗扰动增强
# 使用方法：需要提供预训练的ULMFiT检查点

import argparse
import csv
import itertools
import math
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    BucketBatcher,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    split_indices,
    subset_dataset,
)
from train_ulmfit_scratch import ULMFiTClassifier, evaluate_classifier


def forward_from_emb(model: ULMFiTClassifier, emb: torch.Tensor, x: torch.Tensor, lengths: torch.Tensor):
    out, _, _ = model.encoder(emb, lengths, None)
    mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
    last_idx = (lengths - 1).clamp_min(0).view(-1, 1, 1).expand(-1, 1, out.size(2))
    last = out.gather(1, last_idx).squeeze(1)
    max_pool = out.masked_fill(~mask.unsqueeze(2), -1e4).max(dim=1).values
    mean_pool = (
        out.masked_fill(~mask.unsqueeze(2), 0.0).sum(dim=1)
        / lengths.clamp_min(1).to(out.dtype).unsqueeze(1)
    )
    attn_score = model.attn(out).squeeze(2).masked_fill(~mask, -1e4)
    attn = F.softmax(attn_score, dim=1).unsqueeze(2)
    attn_pool = (out * attn).sum(dim=1)
    return model.head(torch.cat([last, max_pool, mean_pool, attn_pool], dim=1)).squeeze(1)


def normalize_perturbation(d: torch.Tensor, x: torch.Tensor):
    mask = (x != 0).to(d.dtype).unsqueeze(2)
    d = d * mask
    flat = d.view(d.size(0), -1)
    norm = torch.norm(flat, p=2, dim=1, keepdim=True).clamp_min(1e-8)
    return (flat / norm).view_as(d) * mask


def bernoulli_kl_with_logits(base_logits, perturbed_logits):
    p = torch.sigmoid(base_logits.detach()).clamp(1e-6, 1 - 1e-6)
    logp = torch.log(p)
    log1p = torch.log1p(-p)
    logq = F.logsigmoid(perturbed_logits)
    log1q = F.logsigmoid(-perturbed_logits)
    return (p * (logp - logq) + (1 - p) * (log1p - log1q)).mean()


def vat_loss(model, x, lengths, xi, eps):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        emb = model.embedding(x)
        base_logits = forward_from_emb(model, emb, x, lengths)

    d = torch.randn_like(emb)
    d = normalize_perturbation(d, x)
    d.requires_grad_()
    perturbed_logits = forward_from_emb(model, emb + xi * d, x, lengths)
    adv_distance = bernoulli_kl_with_logits(base_logits, perturbed_logits)
    grad = torch.autograd.grad(adv_distance, d, only_inputs=True)[0]
    r_adv = eps * normalize_perturbation(grad.detach(), x)
    perturbed_logits = forward_from_emb(model, emb + r_adv, x, lengths)
    loss = bernoulli_kl_with_logits(base_logits, perturbed_logits)
    if was_training:
        model.train()
    return loss


def calibrate(model, valid_data, batch_size, device):
    probs = predict_probs(model, valid_data, batch_size, device)
    labels = valid_data.y.tolist()
    best = (0.0, 0.5)
    for i in range(1, 1000):
        th = i / 1000
        acc = sum((p >= th) == (y >= 0.5) for p, y in zip(probs, labels)) / len(labels)
        if acc > best[0]:
            best = (acc, th)
    return best


def write_variants(model, test_ids, test_texts, word2idx, cfg, args, device):
    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    probs = predict_probs(model, test_data, args.batch_size, device)
    with open(args.submission, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, prob in zip(test_ids, probs):
            writer.writerow([sample_id, int(prob >= 0.5)])

    order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
    labels = [0] * len(probs)
    for i in order[: len(probs) // 2]:
        labels[i] = 1
    balanced_path = args.submission.replace(".csv", "_balanced.csv")
    with open(balanced_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, label in zip(test_ids, labels):
            writer.writerow([sample_id, label])
    print(f"wrote {args.submission} and {balanced_path}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="runs_ulmfit_continue/ulmfit_clf_best.pt")
    parser.add_argument("--base-final", default="runs_ulmfit_full/final_model.pt")
    parser.add_argument("--out-dir", default="runs_ulmfit_vat")
    parser.add_argument("--submission", default="submission_ulmfit_vat.csv")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--unlabel", default="train_unlabel.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--unlabel-batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--vat-weight", type=float, default=0.8)
    parser.add_argument("--vat-xi", type=float, default=1e-3)
    parser.add_argument("--vat-eps", type=float, default=3.0)
    parser.add_argument("--label-smoothing", type=float, default=0.02)
    parser.add_argument("--weight-decay", type=float, default=2e-4)
    parser.add_argument("--grad-clip", type=float, default=0.25)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2033)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--no-amp", dest="amp", action="store_false")
    parser.add_argument("--max-unlabel", type=int, default=None)
    parser.add_argument("--max-train", type=int, default=None)
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    base = torch.load(args.base_final, map_location="cpu", weights_only=False)
    cfg = dict(checkpoint["args"])
    word2idx = base["word2idx"]
    idx2word = base["idx2word"]

    train_texts, train_labels = read_train(args.train, args.max_train)
    unlabel_texts = read_unlabel(args.unlabel, args.max_unlabel)
    test_ids, test_texts = read_test(args.test)
    full_train = make_dataset(train_texts, train_labels, word2idx, cfg["seq_len"])
    train_idx, valid_idx = split_indices(len(full_train), args.valid_ratio, args.seed)
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)
    unlabel_data = make_dataset(unlabel_texts, None, word2idx, cfg["seq_len"])

    model = ULMFiTClassifier(
        len(idx2word),
        cfg["emb_dim"],
        cfg["hidden_dim"],
        cfg["layers"],
        cfg["dropout"],
        word_dropout=cfg.get("word_dropout", 0.04),
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    train_batcher = BucketBatcher(train_data, args.batch_size, True, args.seed)
    unlabel_batcher = BucketBatcher(unlabel_data, args.unlabel_batch_size, True, args.seed + 77)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.embedding.parameters(), "lr": args.lr * 0.25},
            {"params": model.encoder.parameters(), "lr": args.lr * 0.5},
            {"params": model.attn.parameters(), "lr": args.lr},
            {"params": model.head.parameters(), "lr": args.lr},
        ],
        weight_decay=args.weight_decay,
    )
    total_steps = len(train_batcher) * args.epochs
    warmup = max(1, int(total_steps * 0.08))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    best_acc = -1.0
    best_path = os.path.join(args.out_dir, "ulmfit_vat_best.pt")

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0
        unlabeled_iter = itertools.cycle(iter(unlabel_batcher))
        for batch in train_batcher:
            x, lengths, labels = [item.to(device, non_blocking=True) for item in batch]
            ux, ulengths = [item.to(device, non_blocking=True) for item in next(unlabeled_iter)]
            targets = labels * (1 - 2 * args.label_smoothing) + args.label_smoothing
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp and device.type == "cuda"):
                logits = model(x, lengths)
                sup_loss = F.binary_cross_entropy_with_logits(logits, targets)
            adv_loss = vat_loss(model, ux, ulengths, args.vat_xi, args.vat_eps)
            loss = sup_loss + args.vat_weight * adv_loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total += labels.numel()
                total_loss += loss.item() * labels.numel()
        valid_batcher = BucketBatcher(valid_data, args.batch_size, False, args.seed)
        val_loss, val_acc = evaluate_classifier(model, valid_batcher, device)
        cal_acc, cal_th = calibrate(model, valid_data, args.batch_size, device)
        print(
            f"vat epoch {epoch:02d} train_loss={total_loss/max(1,total):.5f} "
            f"train_acc={total_correct/max(1,total):.4f} val_acc={val_acc:.4f} "
            f"cal_acc={cal_acc:.4f} cal_th={cal_th:.3f}"
        )
        if cal_acc > best_acc:
            best_acc = cal_acc
            torch.save(
                {"model": model.state_dict(), "args": cfg, "calibrated_val_acc": best_acc, "val_acc": val_acc},
                best_path,
            )
            print(f"saved {best_path} cal_acc={best_acc:.4f}")

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    write_variants(model, test_ids, test_texts, word2idx, cfg, args, device)
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": cfg,
            "calibrated_val_acc": best_acc,
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第八部分：自训练模型（来自selftrain_from_checkpoint.py）
# ============================================
# 功能：伪标签自训练（Self-Training with Pseudo Labels）
# 特性：高置信度样本选择、加权损失函数
# 使用方法：需要提供预训练模型的检查点

import argparse
import os
from argparse import Namespace

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch

from train_scratch_lstm import (
    TextLSTMClassifier,
    build_vocab,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    split_indices,
    subset_dataset,
    tokenize,
    train_model,
    write_submission,
)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--out-dir", default="runs_selftrain_fixed")
    parser.add_argument("--submission", default="submission.csv")
    parser.add_argument("--threshold", type=float, default=0.995)
    parser.add_argument("--per-class-limit", type=int, default=8000)
    parser.add_argument("--pseudo-weight", type=float, default=0.12)
    parser.add_argument("--epochs", type=int, default=4)
    parser.add_argument("--lr", type=float, default=2e-4)
    return parser.parse_args()


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    cfg = checkpoint["args"]
    train_texts, train_labels = read_train(cfg.get("train", "train.csv"), cfg.get("max_train"))
    unlabel_texts = read_unlabel(cfg.get("unlabel", "train_unlabel.csv"), cfg.get("max_unlabel"))
    test_ids, test_texts = read_test(cfg.get("test", "test.csv"))

    vocab_texts = train_texts + unlabel_texts
    if cfg.get("include_test_vocab", False):
        vocab_texts += test_texts
    word2idx, idx2word = build_vocab(
        [tokenize(text) for text in vocab_texts],
        cfg.get("min_count", 2),
        cfg.get("max_vocab", 90000),
    )

    full_train = make_dataset(train_texts, train_labels, word2idx, cfg["seq_len"])
    train_idx, valid_idx = split_indices(
        len(full_train), cfg.get("valid_ratio", 0.1), cfg.get("seed", 2026)
    )
    valid_data = subset_dataset(full_train, valid_idx)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TextLSTMClassifier(
        vocab_size=len(idx2word),
        emb_dim=cfg["emb_dim"],
        hidden_dim=cfg["hidden_dim"],
        dropout=cfg["dropout"],
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    print("predicting unlabeled for fixed self-training ...")
    unlabel_data = make_dataset(unlabel_texts, None, word2idx, cfg["seq_len"])
    probs = predict_probs(model, unlabel_data, cfg["batch_size"], device)

    pos = [(i, p) for i, p in enumerate(probs) if p >= args.threshold]
    neg = [(i, p) for i, p in enumerate(probs) if p <= 1.0 - args.threshold]
    pos.sort(key=lambda item: item[1], reverse=True)
    neg.sort(key=lambda item: item[1])
    pos = pos[: args.per_class_limit]
    neg = neg[: args.per_class_limit]
    selected = pos + neg
    print(
        f"pseudo_selected={len(selected)} pos={len(pos)} neg={len(neg)} "
        f"threshold={args.threshold}"
    )

    pseudo_texts = [unlabel_texts[i] for i, _ in selected]
    pseudo_labels = [1.0 if p >= 0.5 else 0.0 for _, p in selected]
    combined_texts = [train_texts[i] for i in train_idx] + pseudo_texts
    combined_labels = [float(train_labels[i]) for i in train_idx] + pseudo_labels
    combined_weights = [1.0] * len(train_idx) + [args.pseudo_weight] * len(pseudo_texts)
    train_plus_pseudo = make_dataset(
        combined_texts,
        combined_labels,
        word2idx,
        cfg["seq_len"],
        combined_weights,
    )

    train_args = Namespace(**cfg)
    train_args.out_dir = args.out_dir
    train_args.epochs = args.epochs
    train_args.lr = cfg.get("lr", 0.0015)
    train_args.pseudo_weight = args.pseudo_weight

    print("fine-tuning with fixed pseudo labels ...")
    self_acc = train_model(
        model,
        train_plus_pseudo,
        valid_data,
        train_args,
        device,
        "selftrain_fixed",
        lr=args.lr,
    )
    best_path = os.path.join(args.out_dir, "selftrain_fixed_best.pt")
    best_acc = checkpoint.get("val_acc", 0.0)
    if self_acc >= best_acc:
        final_path = best_path
        final_acc = self_acc
    else:
        final_path = args.checkpoint
        final_acc = best_acc
        checkpoint = torch.load(final_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model"])

    print(f"using checkpoint={final_path} val_acc={final_acc}")
    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    test_probs = predict_probs(model, test_data, cfg["batch_size"], device)
    write_submission(test_ids, test_probs, args.submission)
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": cfg,
            "val_acc": final_acc,
            "source_checkpoint": final_path,
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"wrote {args.submission}")
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第九部分：继续微调（来自continue_ulmfit_finetune.py）
# ============================================
# 功能：从已有检查点继续微调ULMFiT分类器
# 特性：阈值校准、模型融合
# 使用方法：需要提供ULMFiT预训练的检查点

import argparse
import csv
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch

from train_scratch_lstm import (
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    split_indices,
    subset_dataset,
)
from train_ulmfit_scratch import ULMFiTClassifier, train_classifier


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="runs_ulmfit_full/final_model.pt")
    parser.add_argument("--out-dir", default="runs_ulmfit_continue")
    parser.add_argument("--submission", default="submission.csv")
    parser.add_argument("--clf-epochs", type=int, default=4)
    parser.add_argument("--clf-lr", type=float, default=3e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.03)
    return parser.parse_args()


def calibrate_and_write(model, valid_data, test_ids, test_texts, word2idx, cfg, path, device):
    valid_probs = predict_probs(model, valid_data, cfg["batch_size"], device)
    labels = valid_data.y.tolist()
    best = (0.0, 0.5)
    for i in range(1, 1000):
        th = i / 1000
        acc = sum((p >= th) == (y >= 0.5) for p, y in zip(valid_probs, labels)) / len(labels)
        if acc > best[0]:
            best = (acc, th)

    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    test_probs = predict_probs(model, test_data, cfg["batch_size"], device)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, prob in zip(test_ids, test_probs):
            writer.writerow([sample_id, int(prob >= best[1])])
    print(f"calibrated_val_acc={best[0]:.4f} threshold={best[1]:.3f}")
    print(f"wrote {path}")
    return best


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    cfg = checkpoint["args"]
    cfg = dict(cfg)
    cfg["out_dir"] = args.out_dir
    cfg["clf_epochs"] = args.clf_epochs
    cfg["clf_lr"] = args.clf_lr
    cfg["label_smoothing"] = args.label_smoothing

    train_texts, train_labels = read_train(cfg.get("train", "train.csv"), cfg.get("max_train"))
    test_ids, test_texts = read_test(cfg.get("test", "test.csv"))
    word2idx = checkpoint["word2idx"]
    idx2word = checkpoint["idx2word"]

    full_train = make_dataset(train_texts, train_labels, word2idx, cfg["seq_len"])
    train_idx, valid_idx = split_indices(len(full_train), cfg.get("valid_ratio", 0.1), cfg.get("seed", 2029))
    train_data = subset_dataset(full_train, train_idx)
    valid_data = subset_dataset(full_train, valid_idx)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ULMFiTClassifier(
        len(idx2word),
        cfg["emb_dim"],
        cfg["hidden_dim"],
        cfg["layers"],
        cfg["dropout"],
        word_dropout=cfg.get("word_dropout", 0.04),
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    from argparse import Namespace

    print("continuing ULMFiT classifier fine-tune ...")
    best_acc = train_classifier(model, train_data, valid_data, Namespace(**cfg), device)
    print(f"continued_best_val_acc={best_acc:.4f}")
    best_path = os.path.join(args.out_dir, "ulmfit_clf_best.pt")
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model"])

    cal_acc, threshold = calibrate_and_write(
        model, valid_data, test_ids, test_texts, word2idx, cfg, args.submission, device
    )
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": cfg,
            "val_acc": best_acc,
            "calibrated_val_acc": cal_acc,
            "threshold": threshold,
            "source_checkpoint": args.checkpoint,
        },
        os.path.join(args.out_dir, "final_model.pt"),
    )
    print(f"saved {os.path.join(args.out_dir, 'final_model.pt')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第十部分：全量微调（来自finetune_all_from_checkpoint.py）
# ============================================
# 功能：对所有参数进行微调
# 特性：AdamW优化器、余弦退火学习率调度
# 使用方法：需要提供预训练模型的检查点

import argparse
import math
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.nn import functional as F

from train_scratch_lstm import (
    BucketBatcher,
    EncodedDataset,
    TextLSTMClassifier,
    build_vocab,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    set_seed,
    tokenize,
    weighted_bce,
    write_submission,
)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--submission", default="submission.csv")
    parser.add_argument("--out-checkpoint", default=None)
    return parser.parse_args()


def train_all(model, dataset: EncodedDataset, cfg: dict, epochs: int, lr: float, device):
    batcher = BucketBatcher(dataset, cfg["batch_size"], True, cfg["seed"])
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=cfg.get("weight_decay", 0.0),
        betas=(0.9, 0.999),
    )
    total_steps = max(1, len(batcher) * epochs)
    warmup_steps = max(1, int(total_steps * 0.05))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.2 + 0.8 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    amp_enabled = cfg.get("amp", True) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0
        for batch in batcher:
            x, lengths, labels = tuple(item.to(device, non_blocking=True) for item in batch)
            weights = torch.ones_like(labels)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                logits = model(x, lengths)
                loss = weighted_bce(logits, labels, weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.get("grad_clip", 1.0))
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            with torch.no_grad():
                pred = (torch.sigmoid(logits) >= 0.5).float()
                total_correct += (pred == labels).sum().item()
                total_seen += labels.numel()
                total_loss += loss.item() * labels.numel()

        print(
            f"full epoch {epoch:02d} "
            f"train_loss={total_loss / max(1, total_seen):.5f} "
            f"train_acc={total_correct / max(1, total_seen):.4f}"
        )


def main():
    args = parse_args()
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    cfg = checkpoint["args"]
    set_seed(cfg["seed"])

    train_texts, train_labels = read_train(cfg.get("train", "train.csv"), cfg.get("max_train"))
    unlabel_texts = read_unlabel(cfg.get("unlabel", "train_unlabel.csv"), cfg.get("max_unlabel"))
    test_ids, test_texts = read_test(cfg.get("test", "test.csv"))

    vocab_texts = train_texts + unlabel_texts
    if cfg.get("include_test_vocab", False):
        vocab_texts += test_texts
    word2idx, idx2word = build_vocab(
        [tokenize(text) for text in vocab_texts],
        cfg.get("min_count", 2),
        cfg.get("max_vocab", 90000),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TextLSTMClassifier(
        vocab_size=len(idx2word),
        emb_dim=cfg["emb_dim"],
        hidden_dim=cfg["hidden_dim"],
        dropout=cfg["dropout"],
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    dataset = make_dataset(train_texts, train_labels, word2idx, cfg["seq_len"])
    train_all(model, dataset, cfg, args.epochs, args.lr, device)

    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    probs = predict_probs(model, test_data, cfg["batch_size"], device)
    write_submission(test_ids, probs, args.submission)

    out_checkpoint = args.out_checkpoint or os.path.join(cfg.get("out_dir", "."), "full_finetuned_final.pt")
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": cfg,
            "source_checkpoint": args.checkpoint,
            "full_finetune_epochs": args.epochs,
            "full_finetune_lr": args.lr,
        },
        out_checkpoint,
    )
    print(f"wrote {args.submission}")
    print(f"saved {out_checkpoint}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第十一部分：生成提交（来自make_submission_from_checkpoint.py）
# ============================================
# 功能：从检查点加载模型并生成测试集预测
# 特性：简单直接的预测流程
# 使用方法：需要提供训练好的模型检查点

import argparse
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch

from train_scratch_lstm import (
    TextLSTMClassifier,
    build_vocab,
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    read_unlabel,
    tokenize,
    write_submission,
)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--submission", default="submission.csv")
    return parser.parse_args()


def main():
    args = parse_args()
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    cfg = checkpoint["args"]

    train_texts, _ = read_train(cfg.get("train", "train.csv"), cfg.get("max_train"))
    unlabel_texts = read_unlabel(cfg.get("unlabel", "train_unlabel.csv"), cfg.get("max_unlabel"))
    test_ids, test_texts = read_test(cfg.get("test", "test.csv"))

    vocab_texts = train_texts + unlabel_texts
    if cfg.get("include_test_vocab", False):
        vocab_texts += test_texts
    word2idx, idx2word = build_vocab(
        [tokenize(text) for text in vocab_texts],
        cfg.get("min_count", 2),
        cfg.get("max_vocab", 90000),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TextLSTMClassifier(
        vocab_size=len(idx2word),
        emb_dim=cfg["emb_dim"],
        hidden_dim=cfg["hidden_dim"],
        dropout=cfg["dropout"],
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    probs = predict_probs(model, test_data, cfg["batch_size"], device)
    write_submission(test_ids, probs, args.submission)

    out_dir = cfg.get("out_dir", ".")
    os.makedirs(out_dir, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "args": cfg,
            "val_acc": checkpoint.get("val_acc"),
            "source_checkpoint": args.checkpoint,
        },
        os.path.join(out_dir, "final_model.pt"),
    )
    print(f"wrote {args.submission}")
    print(f"saved {os.path.join(out_dir, 'final_model.pt')}")
    print(f"checkpoint_val_acc={checkpoint.get('val_acc')}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 第十二部分：NB-LSTM混合模型（来自nb_lstm_hybrid.py）
# ============================================
# 功能：朴素贝叶斯与LSTM的集成模型
# 特性：n-gram哈希特征、网格搜索最优融合权重
# 使用方法：需要ULMFiT或LSTM模型的检查点

import argparse
import csv
import math
import os
from typing import Sequence

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch

from train_scratch_lstm import (
    make_dataset,
    predict_probs,
    read_test,
    read_train,
    split_indices,
    subset_dataset,
    tokenize,
)
from train_ulmfit_scratch import ULMFiTClassifier


HASH_BASE = 1000003


def stable_hash_ids(parts: Sequence[int], buckets: int) -> int:
    h = 1469598103934665603
    for part in parts:
        h ^= part + 0x9E3779B97F4A7C15
        h = (h * HASH_BASE) & 0xFFFFFFFFFFFFFFFF
    return h % buckets


def doc_features(text, word2idx, buckets, max_tokens, orders):
    ids = [word2idx.get(token, 1) for token in tokenize(text)[:max_tokens]]
    feats = set()
    for order in orders:
        if len(ids) < order:
            continue
        for i in range(len(ids) - order + 1):
            feats.add(stable_hash_ids(ids[i : i + order], buckets))
    return feats


def build_nb_ratio(texts, labels, word2idx, indices, buckets, max_tokens, orders, alpha):
    pos = [alpha] * buckets
    neg = [alpha] * buckets
    pos_total = alpha * buckets
    neg_total = alpha * buckets
    for idx in indices:
        feats = doc_features(texts[idx], word2idx, buckets, max_tokens, orders)
        if labels[idx] == 1:
            for feat in feats:
                pos[feat] += 1.0
            pos_total += len(feats)
        else:
            for feat in feats:
                neg[feat] += 1.0
            neg_total += len(feats)
    return [math.log((p / pos_total) / (n / neg_total)) for p, n in zip(pos, neg)]


def nb_scores(texts, word2idx, ratio, buckets, max_tokens, orders):
    scores = []
    for text in texts:
        feats = doc_features(text, word2idx, buckets, max_tokens, orders)
        if feats:
            scores.append(sum(ratio[feat] for feat in feats) / math.sqrt(len(feats)))
        else:
            scores.append(0.0)
    return scores


def logit(p):
    p = min(max(p, 1e-6), 1 - 1e-6)
    return math.log(p / (1 - p))


def grid_search(base_logits, nb, labels):
    best = (0.0, 1.0, 0.0, 0.0)
    for nb_scale in [-2, -1.5, -1, -0.75, -0.5, -0.25, 0.25, 0.5, 0.75, 1, 1.5, 2]:
        for base_scale in [0.5, 0.75, 1.0, 1.25, 1.5]:
            scores = [base_scale * b + nb_scale * n for b, n in zip(base_logits, nb)]
            for th_i in range(1, 1000, 2):
                th = (th_i - 500) / 100.0
                acc = sum((s >= th) == (y == 1) for s, y in zip(scores, labels)) / len(labels)
                if acc > best[0]:
                    best = (acc, base_scale, nb_scale, th)
    return best


def write_submission(path, ids, scores, threshold):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, score in zip(ids, scores):
            writer.writerow([sample_id, int(score >= threshold)])


def write_balanced(path, ids, scores):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    labels = [0] * len(scores)
    for i in order[: len(scores) // 2]:
        labels[i] = 1
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for sample_id, label in zip(ids, labels):
            writer.writerow([sample_id, label])


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="runs_ulmfit_continue/ulmfit_clf_best.pt")
    parser.add_argument("--base-final", default="runs_ulmfit_full/final_model.pt")
    parser.add_argument("--submission", default="submission_nb_lstm.csv")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--buckets", type=int, default=1048576)
    parser.add_argument("--max-tokens", type=int, default=1200)
    parser.add_argument("--orders", default="1,2")
    parser.add_argument("--alpha", type=float, default=1.0)
    parser.add_argument("--split-seed", type=int, default=2029)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    return parser.parse_args()


def main():
    args = parse_args()
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    base = torch.load(args.base_final, map_location="cpu", weights_only=False)
    cfg = checkpoint["args"]
    word2idx = base["word2idx"]
    idx2word = base["idx2word"]
    orders = tuple(int(x) for x in args.orders.split(",") if x)

    train_texts, train_labels = read_train(args.train)
    train_idx, valid_idx = split_indices(len(train_texts), args.valid_ratio, args.split_seed)
    print(f"train_split={len(train_idx)} valid={len(valid_idx)}")
    ratio = build_nb_ratio(
        train_texts,
        train_labels,
        word2idx,
        train_idx,
        args.buckets,
        args.max_tokens,
        orders,
        args.alpha,
    )
    valid_texts = [train_texts[i] for i in valid_idx]
    valid_labels = [train_labels[i] for i in valid_idx]
    valid_nb = nb_scores(valid_texts, word2idx, ratio, args.buckets, args.max_tokens, orders)

    valid_data = make_dataset(valid_texts, valid_labels, word2idx, cfg["seq_len"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ULMFiTClassifier(
        len(idx2word),
        cfg["emb_dim"],
        cfg["hidden_dim"],
        cfg["layers"],
        cfg["dropout"],
        word_dropout=cfg.get("word_dropout", 0.04),
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    valid_probs = predict_probs(model, valid_data, cfg["batch_size"], device)
    valid_logits = [logit(p) for p in valid_probs]

    nb_best = (0.0, 0.0)
    for th_i in range(-500, 501):
        th = th_i / 100.0
        acc = sum((s >= th) == (y == 1) for s, y in zip(valid_nb, valid_labels)) / len(valid_labels)
        if acc > nb_best[0]:
            nb_best = (acc, th)
    base_acc = sum((p >= 0.5) == (y == 1) for p, y in zip(valid_probs, valid_labels)) / len(valid_labels)
    mix_best = grid_search(valid_logits, valid_nb, valid_labels)
    print(f"base_acc@0.5={base_acc:.4f}")
    print(f"nb_best_acc={nb_best[0]:.4f} nb_threshold={nb_best[1]:.3f}")
    print(
        f"mix_best_acc={mix_best[0]:.4f} base_scale={mix_best[1]} "
        f"nb_scale={mix_best[2]} threshold={mix_best[3]:.3f}"
    )

    test_ids, test_texts = read_test(args.test)
    test_data = make_dataset(test_texts, None, word2idx, cfg["seq_len"])
    test_probs = predict_probs(model, test_data, cfg["batch_size"], device)
    test_logits = [logit(p) for p in test_probs]
    test_nb = nb_scores(test_texts, word2idx, ratio, args.buckets, args.max_tokens, orders)
    _, base_scale, nb_scale, threshold = mix_best
    test_scores = [base_scale * b + nb_scale * n for b, n in zip(test_logits, test_nb)]
    write_submission(args.submission, test_ids, test_scores, threshold)
    write_balanced(args.submission.replace(".csv", "_balanced.csv"), test_ids, test_scores)
    print(f"wrote {args.submission}")
    print(f"wrote {args.submission.replace('.csv', '_balanced.csv')}")


if __name__ == "__main__":
    main()


## 📝 使用示例

### 示例1：训练基础LSTM分类器
```python
# 首先确保已上传数据文件
# 然后运行以下代码（需要根据实际文件名调整参数）

args = argparse.Namespace(
    train="train.csv",
    unlabel="train_unlabel.csv",
    test="test.csv",
    out_dir="runs_scratch_lstm",
    submission="submission.csv",
    seq_len=320,
    min_count=2,
    max_vocab=90000,
    emb_dim=256,
    hidden_dim=224,
    dropout=0.35,
    batch_size=128,
    epochs=8,
    lr=2e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    warmup_ratio=0.08,
    label_smoothing=0.0,
    valid_ratio=0.1,
    seed=2026,
    amp=True,
    include_test_vocab=True
)

# 调用main函数开始训练
# main()  # 取消注释以运行
```

### 示例2：训练ULMFiT模型
```python
# ULMFiT分为两个阶段：语言模型预训练 + 分类器微调
# 第一阶段：预训练语言模型
args = argparse.Namespace(
    train="train.csv",
    unlabel="train_unlabel.csv",
    test="test.csv",
    out_dir="runs_ulmfit_scratch",
    submission="submission_ulmfit.csv",
    seq_len=512,
    bptt=80,
    min_count=3,
    max_vocab=60000,
    emb_dim=256,
    hidden_dim=256,
    layers=2,
    dropout=0.35,
    word_dropout=0.04,
    lm_batch_size=64,
    batch_size=96,
    lm_epochs=2,
    clf_epochs=6,
    lm_lr=0.0015,
    clf_lr=0.001,
    weight_decay=0.0002,
    alpha=1e-4,
    beta=1e-4,
    grad_clip=0.25,
    label_smoothing=0.04,
    valid_ratio=0.1,
    seed=2029,
    amp=True,
    skip_lm=False,
    skip_predict=False
)

# 调用main函数开始训练
# main()  # 取消注释以运行
```

### 提示
- 每个模型都有独立的参数配置
- 建议先在小数据集上测试代码
- 使用GPU可以显著加速训练过程
- 检查点会保存在指定的out_dir目录中

---
